# 01. Google AI Studio で始める Hello World

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/01_ai_studio_quickstart.ipynb)

本クックブックでは `drawlib.toml` の `[jupyter].setup_packages` 設定により、生成されるすべてのノートブック先頭に `!pip install -q drawlib google-genai pydantic` が自動挿入されるため、Google Colab 上での手動パッケージ追加は不要です。Git リポジトリへ認証情報をハードコードせずに安全に Gemini を呼び出すため、**Google Cloud プロジェクト**に紐付いた API キー（`GEMINI_API_KEY`）を **Google AI Studio** で発行し、Colab 左サイドバーの **シークレット（`🔑` アイコン）** に保存して **「ノートブックからのアクセス（Notebook access）」** を ON に設定します。

![図1-1: Google AI Studio APIキー発行と Colab シークレットによる最短実行フロー](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_aistudio_workflow_ja.png)

*図1-1: Google AI Studio APIキー発行と Colab シークレットによる最短実行フロー*

## 1. Colab シークレットを使った最小構成の Hello World (`userdata` + `genai.Client`)

Google Colab で最初のコードセルを実行する前に、以下の手順（初回のみ）でシークレットを設定します。

1. **Google AI Studio**（`aistudio.google.com/apikey`）を開き、**「API キーを作成（Create API key）」** から既存または新規の **Google Cloud プロジェクト**を選択してキーを発行します。
2. Google Colab 左サイドバーの **シークレット（`🔑`）** アイコンを開き、名前を `GEMINI_API_KEY` としてキーを貼り付け、**「ノートブックからのアクセス」** トグルを ON にします。
3. 実行時に `userdata.get("GEMINI_API_KEY")` でキーを読み込み、`genai.Client` を初期化します。

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="統一された SDK が開発者の生産性を高める理由を2文で説明してください。",
)
print(response.text)

`genai.Client` は `google-genai` SDK 全体で共通のエントリーポイントです。本章で学ぶ `client.models.*` や `client.chats.*` の呼び出しコードは、AI Studio の API キー認証でも Google Cloud Vertex AI の IAM 認証でも一切書き換えることなくそのまま動作します。

## 2. Google Cloud プロジェクト連携：無料枠（Free Tier）と従量課金枠（Paid Tier）

Google AI Studio で発行されるすべての API キーは、背後にある **Google Cloud プロジェクト**に紐付いています。普段から Google Cloud を利用している開発者は、AI Studio のキー作成ダイアログで既存のプロジェクトを選択できるため、API の有効化（`generativelanguage.googleapis.com`）やクォータ監視、請求管理を一元化できます。

- **無料枠 — Free Tier（課金未設定プロジェクト）**: 選択した Google Cloud プロジェクトに **Cloud Billing（請求先アカウント）が紐付いていない場合**、API キーは完全無料の Free Tier として動作します。ハッカソンや学習・プロトタイピングに最適ですが、レート制限（RPM/TPM/RPD）は低めに設定されており、入出力データが Google 製品の品質改善に利用される場合があります（機密データの送信は避けてください）。
- **従量課金枠 — Paid Tier（Cloud Billing 有効化済み）**: 同じ Google Cloud プロジェクトで **Cloud Billing** を有効化すると、同じ `GEMINI_API_KEY` が自動的に従量課金枠（Pay-as-You-Go）へアップグレードされます。Python コードを1行も変更することなく、本番ワークロード向けの高い RPM/TPM クォータと**エンタープライズ水準のデータプライバシー（プロンプトや出力が Google のモデル学習に使用されない保証）**が適用されます。

![図1-2: Google AI Studio 無料枠と Cloud Billing 従量課金枠（Paid Tier）の比較](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_billing_tiers_ja.png)

*図1-2: Google AI Studio 無料枠と Cloud Billing 従量課金枠（Paid Tier）の比較*

## 3. `GenerateContentConfig` と Drawlib によるトークン消費量の可視化

モデルの役割設定（システムプロンプト）や出力の再現性（`temperature`）を制御するには、`types.GenerateContentConfig` を `generate_content` に渡します。また、レスポンスの `response.usage_metadata` からは、入力プロンプト（`prompt_token_count`）、内部思考（`thoughts_token_count`）、最終回答（`candidates_token_count`）の正確なトークン数を取得できます。

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction="あなたは簡潔に回答するクラウドアーキテクトです。短い箇条書き3点で答えてください。",
    temperature=0.2,
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="ノートブックで API キーを安全に管理するベストプラクティスを3つ挙げてください。",
    config=config,
)
print(response.text)

usage = response.usage_metadata
prompt_tokens = usage.prompt_token_count or 0
thoughts_tokens = usage.thoughts_token_count or 0
candidates_tokens = usage.candidates_token_count or 0

print(f"Prompt tokens:     {prompt_tokens}")
print(f"Thoughts tokens:   {thoughts_tokens}")
print(f"Candidates tokens: {candidates_tokens}")

ノートブックには `google-genai` と一緒に `drawlib` もセットアップされているため、`setup(..., cwd=True)` と `BarChart`、`show()` を使って、直前のセルで取得した `usage_metadata` のトークン内訳をそのまま次のセルで可視化できます。

In [ ]:
from drawlib.canvas import setup, show
from drawlib.charts.bar import BarChart
from drawlib.styles import Styles

setup(width=110, height=52, cwd=True)

chart = BarChart(
    axis_line_style=Styles.MutedDashed,
    categories=["入力 (Prompt)", "思考 (Thoughts)", "出力 (Candidates)"],
    width=96.0,
    height=40.0,
    title="Gemini 2.5 Flash — トークン消費量の内訳",
    title_style=Styles.DarkBold.patch(text_size=12.0),
    background_style=Styles.Neutral.patch(shape_r=2.0),
    bar_width_ratio=0.48,
    bar_r=0.8,
    axis_text_style=Styles.Dark.patch(text_size=10.5),
    grid_style=Styles.MutedThin,
    value_text_style=Styles.DarkBold.patch(text_size=10.5),
    value_format="{:.0f}",
)
chart.add_series(
    "トークン数",
    [float(prompt_tokens), float(thoughts_tokens), float(candidates_tokens)],
    style=Styles.PrimaryFlat,
)
chart.configure_y_axis(min_value=0.0)
chart.draw(xy=(7.0, 6.0))

show()

### まとめと次のステップ

- **セキュアで手軽な実行環境**: `GEMINI_API_KEY` を Colab シークレットに保管することで、Git への鍵漏洩を防ぎつつ、Google Cloud プロジェクトの無料枠・従量課金枠をシームレスに使い分けられます。
- **次章（Chapter 02）のプレビュー**: 続く第2章では、後続の Gemini 呼び出しコードを1行も変えずに、`genai.Client` の初期化だけを **Google Cloud Vertex AI**（`vertexai=True`・IAM 認証）へ切り替える方法を解説します。